# 🏆 Module 4.4: Champion/Challenger Workflow with Aliases

**Goal**: Implement a production model management workflow using MLFlow aliases.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

mlflow.autolog(disable=True)
mlflow.set_experiment("04_champion_challenger")

wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)

client = MlflowClient()
MODEL_NAME = "wine-champion-demo"
print("✅ Setup complete!")

## Step 1: Train and Register an Initial Champion

In [ ]:
# Train our first model — this becomes the champion
with mlflow.start_run(run_name="initial_champion"):
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    f1 = f1_score(y_test, model.predict(X_test), average='weighted')
    
    mlflow.log_params({"n_estimators": 100, "max_depth": 5, "model_type": "RandomForest"})
    mlflow.log_metrics({"accuracy": acc, "f1": f1})
    
    result = mlflow.sklearn.log_model(
        model, artifact_path="model",
        registered_model_name=MODEL_NAME
    )
    champion_version = result.registered_model_version
    
    print(f"✅ Registered version {champion_version}: accuracy={acc:.4f}")

In [ ]:
# Set this version as the "champion"
client.set_registered_model_alias(MODEL_NAME, "champion", champion_version)
print(f"🏆 Version {champion_version} is now the CHAMPION!")

## Step 2: Train a Challenger Model

In [ ]:
# Train a potentially better model
with mlflow.start_run(run_name="challenger_model"):
    model = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    )
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    f1 = f1_score(y_test, model.predict(X_test), average='weighted')
    
    mlflow.log_params({
        "n_estimators": 200, "learning_rate": 0.05, 
        "max_depth": 3, "model_type": "GradientBoosting"
    })
    mlflow.log_metrics({"accuracy": acc, "f1": f1})
    
    result = mlflow.sklearn.log_model(
        model, artifact_path="model",
        registered_model_name=MODEL_NAME
    )
    challenger_version = result.registered_model_version
    
    print(f"✅ Registered version {challenger_version}: accuracy={acc:.4f}")

# Set as challenger
client.set_registered_model_alias(MODEL_NAME, "challenger", challenger_version)
print(f"⚔️  Version {challenger_version} is now the CHALLENGER!")

## Step 3: Compare Champion vs Challenger

In [ ]:
# Load both models by alias
champion_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
challenger_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@challenger")

# Compare performance
champion_acc = accuracy_score(y_test, champion_model.predict(X_test))
challenger_acc = accuracy_score(y_test, challenger_model.predict(X_test))

champion_f1 = f1_score(y_test, champion_model.predict(X_test), average='weighted')
challenger_f1 = f1_score(y_test, challenger_model.predict(X_test), average='weighted')

print("📊 Champion vs Challenger:")
print("=" * 40)
print(f"{'Metric':<15} {'Champion':>10} {'Challenger':>12}")
print("-" * 40)
print(f"{'Accuracy':<15} {champion_acc:>10.4f} {challenger_acc:>12.4f}")
print(f"{'F1 Score':<15} {champion_f1:>10.4f} {challenger_f1:>12.4f}")
print()

if challenger_acc > champion_acc:
    print("🎯 Challenger is BETTER! Consider promoting.")
else:
    print("🛡️ Champion holds! Challenger needs more work.")

## Step 4: Promote Challenger to Champion (if better)

In [ ]:
def promote_challenger(model_name, metric_name="accuracy", threshold=0.0):
    """
    Promote challenger to champion if it beats the current champion.
    
    Args:
        model_name: Registered model name
        metric_name: Metric to compare
        threshold: Minimum improvement required (default: any improvement)
    """
    # Load both models
    champion = mlflow.sklearn.load_model(f"models:/{model_name}@champion")
    challenger = mlflow.sklearn.load_model(f"models:/{model_name}@challenger")
    
    # Evaluate
    champion_score = accuracy_score(y_test, champion.predict(X_test))
    challenger_score = accuracy_score(y_test, challenger.predict(X_test))
    
    improvement = challenger_score - champion_score
    
    print(f"Champion {metric_name}: {champion_score:.4f}")
    print(f"Challenger {metric_name}: {challenger_score:.4f}")
    print(f"Improvement: {improvement:+.4f} (threshold: {threshold})")
    
    if improvement > threshold:
        # Get challenger version number
        challenger_info = client.get_model_version_by_alias(model_name, "challenger")
        
        # Promote: set challenger as new champion
        client.set_registered_model_alias(model_name, "champion", challenger_info.version)
        
        # Remove challenger alias
        client.delete_registered_model_alias(model_name, "challenger")
        
        print(f"\n🏆 PROMOTED! Version {challenger_info.version} is the new champion!")
        return True
    else:
        print(f"\n❌ Not promoted. Improvement below threshold.")
        return False

# Try promoting
promoted = promote_challenger(MODEL_NAME)

## Step 5: Production-Style Model Loading

In production, always load by **alias** — never by version number.

In [ ]:
# This is how production code should load models
def predict_wine_quality(features):
    """Production prediction function — always uses the champion model."""
    model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
    return model.predict(features)

# Test it
sample = X_test[:3]
predictions = predict_wine_quality(sample)

print("🔮 Production Predictions:")
for i, (pred, actual) in enumerate(zip(predictions, y_test[:3])):
    print(f"  Sample {i+1}: predicted={wine.target_names[pred]}, actual={wine.target_names[actual]}")

## 🔑 Key Takeaways

1. **Aliases** (`champion`, `challenger`) are mutable pointers to specific versions
2. **Always load by alias** in production: `models:/name@champion`
3. The **Champion/Challenger** pattern enables safe model updates:
   - Train new model → register as version → set as challenger
   - Compare against champion → promote if better
4. **`client.set_registered_model_alias()`** — assign an alias
5. **`client.delete_registered_model_alias()`** — remove an alias
6. **`client.get_model_version_by_alias()`** — get version info from alias

---

## ➡️ Next: `05_exercises.ipynb` — Practice registry workflows